# Retail Conversion Intelligence — TP Final Integrador

**Seminario de Actualización · IFTS N.º 18 · 2026**  
**Autora:** Fernanda Fava

En este trabajo voy a entrenar un modelo para estimar si una sesión de navegación en una tienda online termina en una compra. Después voy a guardar el modelo y utilizarlo desde una API hecha con FastAPI.

**Fuente:** [Online Shoppers Purchasing Intention Dataset — UCI](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset), ID 468. Es un proyecto académico, no una herramienta comercial validada.

## 1. Librerías

Importo las librerías que voy a usar para analizar los datos, entrenar, evaluar y guardar el modelo.

In [ ]:
from pathlib import Path
import os
import sys
import joblib
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)

## 2. Carga del dataset

Voy a trabajar con los datos de UCI. Si el CSV ya está guardado en `data/`, lo leo desde allí. Si todavía no existe, lo descargo y guardo una copia. La variable objetivo es `Revenue` (compra: sí o no).

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebook":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
archivo = DATA_DIR / "online_shoppers_intention.csv"

if archivo.exists():
    df = pd.read_csv(archivo)
    print("CSV local cargado:", archivo)
else:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=468)
    df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
    df.to_csv(archivo, index=False)
    print("Dataset descargado y guardado en:", archivo)

print("Dimensiones:", df.shape)
display(df.head())

## 3. Análisis exploratorio

Voy a revisar las primeras filas, los valores faltantes, las filas repetidas y la proporción de sesiones que terminaron en compra.

In [ ]:
print("VALORES NULOS:", df.isna().sum().sum())
print("REGISTROS DUPLICADOS:", df.duplicated().sum())
print("\nDISTRIBUCIÓN DE COMPRAS:")
print(df["Revenue"].value_counts())
print("\nPORCENTAJE DE COMPRAS:")
print(df["Revenue"].value_counts(normalize=True).mul(100).round(2))

### Interpretación inicial

El dataset tiene **12.330 filas y 18 columnas**. No encontré valores nulos, pero sí **125 registros duplicados**. El **84,53 %** de las sesiones no terminó en compra y el **15,47 %** sí.

Las clases están desbalanceadas, por eso no voy a evaluar el modelo solamente con accuracy. También revisaré F1, precision y recall.

Las filas iguales podrían ser sesiones diferentes porque no tenemos un identificador único. Por ahora **no las elimino**; dejo esta decisión documentada.

In [ ]:
# Ver algunas de las filas repetidas sin modificar el dataset
repetidas = df[df.duplicated(keep=False)]
print("Filas dentro de grupos repetidos:", len(repetidas))
display(repetidas.head(10))

## 4. Selección de variables

Voy a utilizar ocho variables de navegación. No incluyo `PageValues` porque quiero probar el modelo con datos de comportamiento que puedan estar disponibles durante una sesión. `ProductRelated_Duration` se mide en segundos.

In [ ]:
features = [
    "ProductRelated", "ProductRelated_Duration", "BounceRates",
    "ExitRates", "Administrative", "VisitorType", "Weekend", "Month"
]
target = "Revenue"
X = df[features].copy()
y = df[target].astype(int)
print("Variables:", features)
print("Objetivo:", target)
print("Datos de entrada:", X.shape)
print("Valores faltantes en las variables seleccionadas:", X.isna().sum().sum())
print("Meses:", sorted(X["Month"].unique()))
print("Tipos de visitante:", sorted(X["VisitorType"].unique()))

## 5. División de datos y entrenamiento

Reservo el 20 % de los datos para prueba, manteniendo la proporción de compras con `stratify`. Comparo primero dos modelos usando validación cruzada solo sobre entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
numericas = ["ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "Administrative"]
categoricas = ["VisitorType", "Weekend", "Month"]

def preparacion():
    return ColumnTransformer([
        ("numericas", StandardScaler(), numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas)
    ])

modelos = {
    "Regresión logística": Pipeline([
        ("preparacion", preparacion()),
        ("modelo", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preparacion", preparacion()),
        ("modelo", RandomForestClassifier(n_estimators=150, min_samples_leaf=3,
                                          class_weight="balanced_subsample", random_state=42, n_jobs=-1))
    ])
}

validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = {
    nombre: cross_val_score(modelo, X_train, y_train, cv=validacion, scoring="f1").mean()
    for nombre, modelo in modelos.items()
}
display(pd.Series(resultados_cv, name="F1 medio").sort_values(ascending=False))

## 6. Prueba de mejoras

Como hay muchas más sesiones sin compra, voy a probar otras dos configuraciones. Comparo el F1 medio con validación cruzada sobre los datos de entrenamiento. No uso el conjunto de prueba para elegir el modelo.

In [ ]:
# Dos configuraciones adicionales
modelos_nuevos = {
    "Logística sin balanceo": Pipeline([
        ("preparacion", preparacion()),
        ("modelo", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Random Forest hojas 5": Pipeline([
        ("preparacion", preparacion()),
        ("modelo", RandomForestClassifier(
            n_estimators=150, min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=42, n_jobs=-1
        ))
    ])
}

modelos.update(modelos_nuevos)

for nombre, modelo in modelos_nuevos.items():
    resultados_cv[nombre] = cross_val_score(
        modelo, X_train, y_train, cv=validacion, scoring="f1"
    ).mean()

comparacion = pd.Series(resultados_cv, name="F1 medio").sort_values(ascending=False)
display(comparacion)

## 7. Evaluación final

Elijo la configuración con mejor F1 medio en validación cruzada. Después la entreno con todos los datos de entrenamiento y la evalúo una sola vez con el conjunto de prueba. También reviso los falsos positivos y negativos.

In [ ]:
nombre_ganador = max(resultados_cv, key=resultados_cv.get)
modelo_final = modelos[nombre_ganador].fit(X_train, y_train)
predicciones = modelo_final.predict(X_test)
probabilidades = modelo_final.predict_proba(X_test)[:, 1]

metricas = {
    "modelo": nombre_ganador,
    "f1_test": float(f1_score(y_test, predicciones)),
    "precision_test": float(precision_score(y_test, predicciones, zero_division=0)),
    "recall_test": float(recall_score(y_test, predicciones, zero_division=0)),
    "roc_auc_test": float(roc_auc_score(y_test, probabilidades))
}
print("Modelo elegido:", nombre_ganador)
print("Matriz de confusión:")
print(confusion_matrix(y_test, predicciones))
print(classification_report(y_test, predicciones, zero_division=0))
print(metricas)

## 8. Guardado del modelo

Guardo el preprocesamiento y el clasificador juntos en `modelo.joblib`, para que la API reciba las variables originales. Registro la versión de scikit-learn y el tamaño del archivo.

In [ ]:
ruta_modelo = PROJECT_DIR / "app" / "modelo" / "modelo.joblib"
ruta_modelo.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_final, ruta_modelo)
print("Modelo guardado en:", ruta_modelo)
print("Versión de scikit-learn:", sklearn.__version__)
print("Tamaño en MB:", round(os.path.getsize(ruta_modelo) / 1e6, 3))

## 9. Conclusiones

**Resultados de la primera prueba (antes de comparar las cuatro configuraciones):** Random Forest obtuvo F1 = 0,357, recall = 0,356 y ROC-AUC = 0,755 para la clase compra. Detectó 136 de 382 compras y dejó sin detectar 246. Por eso, la exactitud general por sí sola no alcanza para evaluar este dataset desbalanceado.

**Pendiente:** volver a ejecutar todas las celdas y anotar aquí los resultados reales del modelo final. Las métricas anteriores son solo una referencia inicial, no el resultado de las mejoras.

Usé IA generativa como ayuda para organizar el notebook y revisar el código. Los datos y métricas se obtienen al ejecutar el proyecto.